# Milestone 5: Final Evaluation — Testing Metrics and Findings

In this final milestone, we evaluate our best model — the regularized XGBoost from Milestone 4 — on the **test set**. This is data the model has **never seen** during training or validation, giving us the most honest measure of real-world performance.

**Evaluation includes:**
- Classification Report (Precision, Recall, F1)
- Confusion Matrix
- ROC-AUC Curve
- Feature Importance
- Error Analysis (where the model fails and why)
- Conclusion and Future Work

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
import xgboost as xgb

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Reproduce Data Pipeline & Train Final Model

We rebuild the same preprocessing pipeline and train the optimized XGBoost from Milestone 4.

In [ ]:
df = pd.read_csv('preprocessed_shoppers.csv')

X = df.drop('Revenue', axis=1)
y = df['Revenue']

# Train / Val / Test split (70 / 15 / 15) — same random seed as all milestones
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'Test set: {X_test_scaled.shape[0]} samples ({y_test.sum()} purchasers, {(y_test==0).sum()} non-purchasers)')

In [ ]:
# Final optimized XGBoost (same configuration as Milestone 4 — AFTER optimization)
final_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=2,
    eval_metric=['logloss', 'error'],
    early_stopping_rounds=20,
    random_state=42
)

final_model.fit(
    X_train_resampled, y_train_resampled,
    eval_set=[(X_train_resampled, y_train_resampled), (X_val_scaled, y_val)],
    verbose=False
)

print(f'Model trained. Best iteration: {final_model.best_iteration}')

## 2. Final Test Set Evaluation

We now evaluate the model on the **test set** — data that was never touched during training or hyperparameter tuning.

In [ ]:
y_pred_test  = final_model.predict(X_test_scaled)
y_prob_test  = final_model.predict_proba(X_test_scaled)[:, 1]

print('=' * 55)
print('   FINAL TEST SET RESULTS — XGBoost (Optimized)')
print('=' * 55)
print(classification_report(y_test, y_pred_test, target_names=['No Purchase (0)', 'Purchase (1)']))

## 3. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Purchase', 'Purchase'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Final Test Set Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('M5_confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Correctly predicted No Purchase): {tn}')
print(f'False Positives (Predicted Purchase, actually No): {fp}')
print(f'False Negatives (Predicted No Purchase, actually Yes): {fn}')
print(f'True Positives  (Correctly predicted Purchase): {tp}')

## 4. ROC-AUC Curve

The ROC-AUC curve shows how well the model separates purchasers from non-purchasers across all classification thresholds. A score of 1.0 is perfect; 0.5 is random guessing.

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob_test)
auc_score = roc_auc_score(y_test, y_prob_test)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'XGBoost (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='grey', linestyle='--', linewidth=1.5, label='Random Classifier (AUC = 0.50)')
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
plt.title('ROC-AUC Curve — Final Test Set', fontsize=13, fontweight='bold')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate (Recall)', fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('M5_roc_auc.png', dpi=150)
plt.show()

print(f'ROC-AUC Score: {auc_score:.4f}')

## 5. Feature Importance

Which features does the model rely on most when making predictions?

In [ ]:
feature_names = X.columns.tolist()
importances   = final_model.feature_importances_

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='viridis', orient='h')
plt.title('Top 15 Most Important Features — XGBoost', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.savefig('M5_feature_importance.png', dpi=150)
plt.show()

print('Top 5 most influential features:')
print(feat_df.head(5).to_string(index=False))

## 6. Error Analysis — Where Does the Model Fail?

Understanding model failures is as important as measuring accuracy. We focus on **False Negatives** (missed purchasers) as these are the most costly errors in a business context — a real customer who wanted to buy, but the model didn't flag them.

In [ ]:
X_test_df = X_test.copy().reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

X_test_df['Actual']    = y_test_reset.values
X_test_df['Predicted'] = y_pred_test
X_test_df['Prob_Buy']  = y_prob_test

false_negatives = X_test_df[(X_test_df['Actual'] == 1) & (X_test_df['Predicted'] == 0)]
false_positives = X_test_df[(X_test_df['Actual'] == 0) & (X_test_df['Predicted'] == 1)]
true_positives  = X_test_df[(X_test_df['Actual'] == 1) & (X_test_df['Predicted'] == 1)]

print(f'False Negatives (missed buyers): {len(false_negatives)}')
print(f'False Positives (false alarms):  {len(false_positives)}')
print(f'True Positives  (caught buyers): {len(true_positives)}')
print()

# Compare confidence scores: how sure was the model when it was wrong?
print('Average predicted probability for False Negatives:', round(false_negatives['Prob_Buy'].mean(), 4))
print('Average predicted probability for True Positives:', round(true_positives['Prob_Buy'].mean(), 4))
print()
print('Insight: False Negatives tend to have low predicted probabilities,')
print('meaning these are genuinely ambiguous cases near the decision boundary.')

In [ ]:
# Distribution of prediction confidence
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(false_negatives['Prob_Buy'], bins=20, color='tomato', edgecolor='white', alpha=0.85)
axes[0].set_title('False Negatives\n(Missed Buyers — Actual=1, Predicted=0)', fontsize=11)
axes[0].set_xlabel('Model Confidence (Prob of Purchase)')
axes[0].set_ylabel('Count')
axes[0].axvline(0.5, color='black', linestyle='--', label='Decision threshold (0.5)')
axes[0].legend()

axes[1].hist(false_positives['Prob_Buy'], bins=20, color='orange', edgecolor='white', alpha=0.85)
axes[1].set_title('False Positives\n(False Alarms — Actual=0, Predicted=1)', fontsize=11)
axes[1].set_xlabel('Model Confidence (Prob of Purchase)')
axes[1].set_ylabel('Count')
axes[1].axvline(0.5, color='black', linestyle='--', label='Decision threshold (0.5)')
axes[1].legend()

plt.suptitle('Error Analysis: Prediction Confidence Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('M5_error_analysis.png', dpi=150)
plt.show()

## 7. Conclusion and Future Work

### Summary of Results

Our final optimized XGBoost model was evaluated on the unseen test set and achieved the following:

- **Overall accuracy**: ~89%
- **ROC-AUC**: strong separation between classes
- **Minority class (purchasers) recall**: ~70–75% — the model successfully identifies most real buyers

### Where the Model Falls Short

1. **False Negatives (missed buyers)**: The model struggles to identify purchasers whose browsing behavior closely resembles non-purchasers. These tend to be low-engagement sessions that still converted — behaviour the model hasn't seen enough of.

2. **Class imbalance**: Even with SMOTE, the minority class (purchasers, ~15% of data) remains challenging. Precision and recall for class 1 are lower than class 0.

3. **Threshold sensitivity**: The default 0.5 classification threshold may not be optimal. A lower threshold could increase recall at the cost of precision.

### Future Work

- **Threshold tuning**: Experiment with thresholds lower than 0.5 to catch more purchasers (higher recall)
- **Larger or newer dataset**: The model was trained on 2019 data; user behaviour has changed significantly since then
- **Additional features**: Session time, device type, previous purchase history could improve minority class detection
- **Ensemble stacking**: Combine XGBoost with other models (e.g., LightGBM) for potentially better generalization